In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

## DataSets & DataLoader & Transform

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from datasets import load_dataset
#image => scale(0,1) => normalize => (-1,1)

# 1. Define your transformation pipeline (exactly as you wrote it)
transform = transforms.Compose([
    transforms.ToTensor(), # Scales PIL Image to [0.0, 1.0]
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Shifts to [-1.0, 1.0]
])

# 2. Download the datasets from Hugging Face (Lightning fast CDN)
trainset_hf = load_dataset("uoft-cs/cifar10", split="train")
testset_hf = load_dataset("uoft-cs/cifar10", split="test")

C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 3. Create a processing function to apply the transformations
def transform_pipeline(examples):
    # Hugging face stores the images under the 'img' key
    # We apply the torchvision transform to each image in the batch
    examples["pixel_values"] = [transform(image.convert("RGB")) for image in examples["img"]]
    return examples

# 4. Bind the transformations dynamically
trainset_hf.set_transform(transform_pipeline)
testset_hf.set_transform(transform_pipeline)

# 5. Define a custom collate function for PyTorch DataLoader
# This organizes the dataset batches into PyTorch Tensors automatically
def collate_fn(examples):
    images = torch.stack([x["pixel_values"] for x in examples])
    labels = torch.tensor([x["label"] for x in examples])
    return images, labels

# 6. Initialize your DataLoaders
train_loader = DataLoader(trainset_hf, batch_size=64, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(testset_hf, batch_size=64, shuffle=False, collate_fn=collate_fn)

# --- Verification Check ---
# Pull one batch to verify the shapes and value scales
for images, labels in train_loader:
    print("Images batch shape:", images.shape) # Expected: torch.Size([64, 3, 32, 32])
    print("Labels batch shape:", labels.shape) # Expected: torch.Size([64])
    print("Min value:", images.min().item())   # Expected: Approx -1.0
    print("Max value:", images.max().item())   # Expected: Approx 1.0
    break


Images batch shape: torch.Size([64, 3, 32, 32])
Labels batch shape: torch.Size([64])
Min value: -1.0
Max value: 1.0


## Build Convolutional Neural Network

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) #flatening
        x = self.fc_layers(x)
        return x

In [5]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Train the CNN

In [6]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()
    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(train_loader)}")


epoch = 1/10 & loss = 1.3915574751089297
epoch = 2/10 & loss = 0.9572616771358968
epoch = 3/10 & loss = 0.7671546661259269
epoch = 4/10 & loss = 0.6285494611315106
epoch = 5/10 & loss = 0.5200896622884609
epoch = 6/10 & loss = 0.42455577915129455
epoch = 7/10 & loss = 0.3379508441747607
epoch = 8/10 & loss = 0.2649222238851554
epoch = 9/10 & loss = 0.20526291245161116
epoch = 10/10 & loss = 0.16122126389209115


## Evaluation of CNN

In [8]:
correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
print(f"accuracy = {(correct_labels/total_labels) * 100}")

accuracy = 74.41
